Data Cleaning

In [ ]:
# =============================================
# STEP 0: Import Required Libraries
# =============================================
import pandas as pd
import numpy as np
from datetime import datetime

# For text cleaning
import re


Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))

df.head()


Saving Project Management Dataset.csv to Project Management Dataset (1).csv


,Project Name,Project Description,Project Type,Project Manager,Region,Department,Project Cost,Project Benefit,Complexity,Status,Completion%,Phase,Year,Month,Start Date,End Date
0,Rhinestone,Associations Now Is A Casual Game To Teach You...,INCOME GENERATION,Yael Wilcox,North,Admin & BI,"3,648,615.00","8,443,980.00",High,In - Progress,77%,Phase 4 - Implement,2021,2,2/1/2021,6/1/2021
1,A Triumph Of Softwares,Is A Fully Managed Content Marketing Software ...,INCOME GENERATION,Brenda Chandler,West,eCommerce,"4,018,835.00","9,012,225.00",High,Cancelled,80%,Phase 2 - Develop,2021,3,3/1/2021,6/1/2021
2,The Blue Bird,Most Content Marketers Know The Golden Rule: Y...,INCOME GENERATION,Nyasia Hunter,North,Warehouse,"4,285,483.00","9,078,339.00",High,Completed,100%,Phase 4 - Implement,2021,3,3/1/2021,6/1/2021
3,Remembering Our Ancestors,"Utilize And Utilizes (Verb Form) The Open, Inc...",PROCESS IMPROVEMENT,Brenda Chandler,East,Sales and Marketing,"5,285,864.00","8,719,006.00",High,Cancelled,75%,Phase 5 - Measure,2021,3,3/1/2021,6/1/2021
4,Skyhawks,Is A Solution For Founders Who Want To Win At ...,WORKING CAPITAL IMPROVEMENT,Jaylyn Mckenzie,East,eCommerce,"5,785,601.00","8,630,148.00",High,Completed,100%,Phase 1 - Explore,2021,3,3/1/2021,6/1/2021


In [ ]:
# =============================================
# STEP 1: Examine Data Types and Missing Values
# =============================================
print(df.info())
print(df.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Project Name         99 non-null     object
 1   Project Description  99 non-null     object
 2   Project Type         99 non-null     object
 3   Project Manager      99 non-null     object
 4   Region               99 non-null     object
 5   Department           99 non-null     object
 6    Project Cost        99 non-null     object
 7    Project Benefit     99 non-null     object
 8   Complexity           99 non-null     object
 9   Status               99 non-null     object
 10  Completion%          99 non-null     object
 11  Phase                99 non-null     object
 12  Year                 99 non-null     int64 
 13  Month                99 non-null     int64 
 14  Start Date           99 non-null     object
 15  End Date             99 non-null     object
dtypes: int64(2

In [ ]:
# =============================================
# STEP 2: Convert Financial Columns to Numeric
# =============================================
# Remove commas and convert to float
df[' Project Cost '] = df[' Project Cost '].replace({',': ''}, regex=True).astype(float)
df[' Project Benefit '] = df[' Project Benefit '].replace({',': ''}, regex=True).astype(float)

# Check
df[[' Project Cost ', ' Project Benefit ']].head()


,Project Cost,Project Benefit
0,3648615.0,8443980.0
1,4018835.0,9012225.0
2,4285483.0,9078339.0
3,5285864.0,8719006.0
4,5785601.0,8630148.0


In [ ]:
# =============================================
# STEP 3: Convert Completion% to Numeric
# =============================================
df['Completion%'] = df['Completion%'].str.replace('%', '', regex=False).astype(float)
df['Completion%'] = df['Completion%'].clip(lower=0, upper=100)  # Ensure valid range

df['Completion%'].head()


,Completion%
0,77.0
1,80.0
2,100.0
3,75.0
4,100.0


In [ ]:
# =============================================
# STEP 4: Convert Dates to Datetime
# =============================================
df['Start Date'] = pd.to_datetime(df['Start Date'], errors='coerce', format='%m/%d/%Y')
df['End Date'] = pd.to_datetime(df['End Date'], errors='coerce', format='%m/%d/%Y')

# Check for invalid dates
print(df[df['Start Date'].isna() | df['End Date'].isna()])


Empty DataFrame
Columns: [Project Name, Project Description, Project Type, Project Manager, Region, Department,  Project Cost ,  Project Benefit , Complexity, Status, Completion%, Phase, Year, Month, Start Date, End Date]
Index: []


In [ ]:
# =============================================
# STEP 5: Standardize Categorical Columns
# =============================================
categorical_cols = ['Project Type', 'Project Manager', 'Region', 'Department',
                    'Complexity', 'Status', 'Phase']

# Strip whitespace and capitalize consistently
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Optional: check unique values
for col in categorical_cols:
    print(f"{col} unique values:\n{df[col].unique()}\n")


Project Type unique values:
['Income Generation' 'Process Improvement' 'Working Capital Improvement'
 'Cost Reduction']

Project Manager unique values:
['Yael Wilcox' 'Brenda Chandler' 'Nyasia Hunter' 'Jaylyn Mckenzie'
 'Kamari Norris' 'Aleena Khan' 'Deacon Delacruz']

Region unique values:
['North' 'West' 'East' 'South']

Department unique values:
['Admin & Bi' 'Ecommerce' 'Warehouse' 'Sales And Marketing' 'Supply Chain']

Complexity unique values:
['High' 'Medium' 'Low']

Status unique values:
['In - Progress' 'Cancelled' 'Completed' 'On - Hold']

Phase unique values:
['Phase 4 - Implement' 'Phase 2 - Develop' 'Phase 5 - Measure'
 'Phase 1 - Explore' 'Phase 3 - Plan']



In [ ]:
# =============================================
# STEP 6: Handle Missing Values
# =============================================
# Example strategy:

# Text/ID columns: fill missing with "Unknown"
text_cols = ['Project Name', 'Project Description', 'Project Manager']
for col in text_cols:
    df[col] = df[col].fillna('Unknown')

# Categorical columns: fill with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Numeric columns: fill with median
numeric_cols = [' Project Cost ', ' Project Benefit ', 'Completion%']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Dates: optional, could drop rows if Start/End missing
df = df.dropna(subset=['Start Date', 'End Date'])


In [ ]:
# =============================================
# STEP 7: Check for Duplicates
# =============================================
# Check duplicates based on Project Name + Year
duplicates = df[df.duplicated(subset=['Project Name', 'Year'], keep=False)]
print("Duplicate projects:", duplicates)

# Optionally drop duplicates
df = df.drop_duplicates(subset=['Project Name', 'Year'])


Duplicate projects: Empty DataFrame
Columns: [Project Name, Project Description, Project Type, Project Manager, Region, Department,  Project Cost ,  Project Benefit , Complexity, Status, Completion%, Phase, Year, Month, Start Date, End Date]
Index: []


In [ ]:
# =============================================
# STEP 8: Logical Consistency Checks
# =============================================

# 1. Project Benefit should be >= Project Cost
df.loc[df[' Project Benefit '] < df[' Project Cost '], ['Project Name', ' Project Cost ', ' Project Benefit ']]

# Optionally flag these for review
df['Benefit_Check'] = np.where(df[' Project Benefit '] >= df[' Project Cost '], 'OK', 'Review')

# 2. End Date >= Start Date
df['Duration_Days'] = (df['End Date'] - df['Start Date']).dt.days
df = df[df['Duration_Days'] >= 0]

# 3. Month range 1-12
df = df[(df['Month'] >= 1) & (df['Month'] <= 12)]

# 4. Year range 2021-2025
df = df[(df['Year'] >= 2021) & (df['Year'] <= 2025)]


In [ ]:
# =============================================
# STEP 9: Clean Text Columns for NLP / Analysis
# =============================================
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)  # normalize spaces
    return text.strip()

df['Project Description_Clean'] = df['Project Description'].apply(clean_text)
df['Project Name_Clean'] = df['Project Name'].apply(clean_text)

df[['Project Name', 'Project Name_Clean', 'Project Description_Clean']].head()


,Project Name,Project Name_Clean,Project Description_Clean
0,Rhinestone,rhinestone,associations now is a casual game to teach you...
1,A Triumph Of Softwares,a triumph of softwares,is a fully managed content marketing software ...
2,The Blue Bird,the blue bird,most content marketers know the golden rule yo...
3,Remembering Our Ancestors,remembering our ancestors,utilize and utilizes verb form the open inclus...
4,Skyhawks,skyhawks,is a solution for founders who want to win at ...


In [ ]:
# =============================================
# STEP 10: Encode Categorical Columns for Analysis
# =============================================
# Option 1: Label Encoding
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col + '_Encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

df.head()


,Project Name,Project Description,Project Type,Project Manager,Region,Department,Project Cost,Project Benefit,Complexity,Status,Completion%,Phase,Year,Month,Start Date,End Date,Benefit_Check,Duration_Days,Project Description_Clean,Project Name_Clean,Project Type_Encoded,Project Manager_Encoded,Region_Encoded,Department_Encoded,Complexity_Encoded,Status_Encoded,Phase_Encoded
0,Rhinestone,Associations Now Is A Casual Game To Teach You...,Income Generation,Yael Wilcox,North,Admin & Bi,3648615.0,8443980.0,High,In - Progress,77.0,Phase 4 - Implement,2021,2,2021-02-01,2021-06-01,OK,120,associations now is a casual game to teach you...,rhinestone,1,6,1,0,0,2,3
1,A Triumph Of Softwares,Is A Fully Managed Content Marketing Software ...,Income Generation,Brenda Chandler,West,Ecommerce,4018835.0,9012225.0,High,Cancelled,80.0,Phase 2 - Develop,2021,3,2021-03-01,2021-06-01,OK,92,is a fully managed content marketing software ...,a triumph of softwares,1,1,3,1,0,0,1
2,The Blue Bird,Most Content Marketers Know The Golden Rule: Y...,Income Generation,Nyasia Hunter,North,Warehouse,4285483.0,9078339.0,High,Completed,100.0,Phase 4 - Implement,2021,3,2021-03-01,2021-06-01,OK,92,most content marketers know the golden rule yo...,the blue bird,1,5,1,4,0,1,3
3,Remembering Our Ancestors,"Utilize And Utilizes (Verb Form) The Open, Inc...",Process Improvement,Brenda Chandler,East,Sales And Marketing,5285864.0,8719006.0,High,Cancelled,75.0,Phase 5 - Measure,2021,3,2021-03-01,2021-06-01,OK,92,utilize and utilizes verb form the open inclus...,remembering our ancestors,2,1,0,2,0,0,4
4,Skyhawks,Is A Solution For Founders Who Want To Win At ...,Working Capital Improvement,Jaylyn Mckenzie,East,Ecommerce,5785601.0,8630148.0,High,Completed,100.0,Phase 1 - Explore,2021,3,2021-03-01,2021-06-01,OK,92,is a solution for founders who want to win at ...,skyhawks,3,3,0,1,0,1,0


In [ ]:
# =============================================
# STEP 11: Derived Columns
# =============================================
# 1. Net_Profit
df['Net_Profit'] = df[' Project Benefit '] - df[' Project Cost ']

# 2. Quarter
df['Quarter'] = df['Month'].apply(lambda x: (x-1)//3 + 1)

# 3. Duration in months
df['Duration_Months'] = df['Duration_Days'] / 30

df[['Project Name', 'Duration_Days', 'Duration_Months', 'Net_Profit', 'Quarter']].head()


,Project Name,Duration_Days,Duration_Months,Net_Profit,Quarter
0,Rhinestone,120,4.000000,4795365.0,1
1,A Triumph Of Softwares,92,3.066667,4993390.0,1
2,The Blue Bird,92,3.066667,4792856.0,1
3,Remembering Our Ancestors,92,3.066667,3433142.0,1
4,Skyhawks,92,3.066667,2844547.0,1


In [ ]:
# =============================================
# STEP 12: Outlier Detection (Optional)
# =============================================
# Simple example using IQR
numeric_features = [' Project Cost ', ' Project Benefit ', 'Completion%', 'Duration_Days', 'Net_Profit']
for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    df[col + '_Outlier'] = ((df[col] < lower) | (df[col] > upper))

df.head()


,Project Name,Project Description,Project Type,Project Manager,Region,Department,Project Cost,Project Benefit,Complexity,Status,Completion%,Phase,Year,Month,Start Date,End Date,Benefit_Check,Duration_Days,Project Description_Clean,Project Name_Clean,Project Type_Encoded,Project Manager_Encoded,Region_Encoded,Department_Encoded,Complexity_Encoded,Status_Encoded,Phase_Encoded,Net_Profit,Quarter,Duration_Months,Project Cost _Outlier,Project Benefit _Outlier,Completion%_Outlier,Duration_Days_Outlier,Net_Profit_Outlier
0,Rhinestone,Associations Now Is A Casual Game To Teach You...,Income Generation,Yael Wilcox,North,Admin & Bi,3648615.0,8443980.0,High,In - Progress,77.0,Phase 4 - Implement,2021,2,2021-02-01,2021-06-01,OK,120,associations now is a casual game to teach you...,rhinestone,1,6,1,0,0,2,3,4795365.0,1,4.000000,False,False,False,True,False
1,A Triumph Of Softwares,Is A Fully Managed Content Marketing Software ...,Income Generation,Brenda Chandler,West,Ecommerce,4018835.0,9012225.0,High,Cancelled,80.0,Phase 2 - Develop,2021,3,2021-03-01,2021-06-01,OK,92,is a fully managed content marketing software ...,a triumph of softwares,1,1,3,1,0,0,1,4993390.0,1,3.066667,False,False,False,False,False
2,The Blue Bird,Most Content Marketers Know The Golden Rule: Y...,Income Generation,Nyasia Hunter,North,Warehouse,4285483.0,9078339.0,High,Completed,100.0,Phase 4 - Implement,2021,3,2021-03-01,2021-06-01,OK,92,most content marketers know the golden rule yo...,the blue bird,1,5,1,4,0,1,3,4792856.0,1,3.066667,False,False,False,False,False
3,Remembering Our Ancestors,"Utilize And Utilizes (Verb Form) The Open, Inc...",Process Improvement,Brenda Chandler,East,Sales And Marketing,5285864.0,8719006.0,High,Cancelled,75.0,Phase 5 - Measure,2021,3,2021-03-01,2021-06-01,OK,92,utilize and utilizes verb form the open inclus...,remembering our ancestors,2,1,0,2,0,0,4,3433142.0,1,3.066667,False,False,False,False,False
4,Skyhawks,Is A Solution For Founders Who Want To Win At ...,Working Capital Improvement,Jaylyn Mckenzie,East,Ecommerce,5785601.0,8630148.0,High,Completed,100.0,Phase 1 - Explore,2021,3,2021-03-01,2021-06-01,OK,92,is a solution for founders who want to win at ...,skyhawks,3,3,0,1,0,1,0,2844547.0,1,3.066667,False,False,False,False,False


In [ ]:
# =============================================
# STEP 13: Save Cleaned Dataset
# =============================================
df.to_csv('cleaned_project_dataset.csv', index=False)
print("Cleaned dataset saved successfully.")


Cleaned dataset saved successfully.
